# The wave equation: a hyperbolic PDE

A vibrating string with fixed ends, $u_{tt}=c^2u_{xx}$ on $x\in[0,1]$, $c=1$, released from a
single-mode shape at rest:
$$u(0,t)=u(1,t)=0,\qquad u(x,0)=\sin\pi x,\qquad u_t(x,0)=0,$$
whose exact solution is the standing wave $u(x,t)=\sin\pi x\,\cos\pi t$. Unlike the elliptic
conduction problem, this is *hyperbolic* and second order in time---an oscillation the network must
carry through two full periods. Runs in a couple of minutes on a CPU.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
PI = np.pi
C, T = 1.0, 2.0                       # wave speed; integrate over two periods
def exact(x, t): return torch.sin(PI*x)*torch.cos(PI*C*t)

In [ ]:
# --- network and the physics-informed loss (soft BC + IC) ---
net = nn.Sequential(nn.Linear(2,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1))
opt = torch.optim.Adam(net.parameters(), 2e-3)
def u(x,t): return net(torch.cat([x,t],1))
def d(f,x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

t0 = time.perf_counter(); EPOCHS = 12000
for e in range(EPOCHS):
    if e == 8000:
        for g in opt.param_groups: g['lr'] = 5e-4
    opt.zero_grad()
    x = torch.rand(3000,1,requires_grad=True); t = (torch.rand(3000,1)*T).requires_grad_(True)
    uu = u(x,t); uxx = d(d(uu,x),x); utt = d(d(uu,t),t)
    res = ((utt - C**2*uxx)**2).mean()                          # wave residual
    tb = torch.rand(400,1)*T
    bc = (u(torch.zeros_like(tb),tb)**2).mean() + (u(torch.ones_like(tb),tb)**2).mean()
    xi = torch.rand(400,1,requires_grad=True); ti = torch.zeros_like(xi).requires_grad_(True)
    ui = u(xi,ti)
    ic = ((ui-torch.sin(PI*xi))**2).mean() + (d(ui,ti)**2).mean()   # displacement + velocity
    loss = res + 10*bc + 10*ic
    loss.backward(); opt.step()
print(f'trained {EPOCHS} in {time.perf_counter()-t0:.0f}s, loss {loss.item():.2e}')

In [ ]:
# --- accuracy vs the exact standing wave ---
n = 200
xs = torch.linspace(0,1,n).reshape(-1,1); ts = torch.linspace(0,T,n).reshape(-1,1)
X, Tt = torch.meshgrid(xs.ravel(), ts.ravel(), indexing='ij')
Xf, Tf = X.reshape(-1,1), Tt.reshape(-1,1)
with torch.no_grad():
    U = u(Xf,Tf).reshape(n,n); Ue = exact(Xf,Tf).reshape(n,n)
relL2 = torch.sqrt(((U-Ue)**2).mean()/(Ue**2).mean()).item()
print(f'rel L2 over the space-time domain (two periods) = {relL2:.3f}')

In [ ]:
# --- figure: space-time fields, snapshots, error history ---
plt.rcParams.update({'figure.dpi':120,'font.size':15,'axes.titlesize':14,'axes.labelsize':15,
    'xtick.labelsize':13,'ytick.labelsize':13,'legend.fontsize':11})
fig, ax = plt.subplots(2,2,figsize=(10.5,8.6)); a = ax.ravel()
xg = xs.numpy().ravel(); tg = ts.numpy().ravel()
c0 = a[0].contourf(tg,xg,U.numpy(),21,cmap='RdBu_r'); plt.colorbar(c0,ax=a[0])
a[0].set_title('PINN  $u(x,t)$'); a[0].set_xlabel('t'); a[0].set_ylabel('x')
c1 = a[1].contourf(tg,xg,Ue.numpy(),21,cmap='RdBu_r'); plt.colorbar(c1,ax=a[1])
a[1].set_title('exact  $\\sin\\pi x\\,\\cos\\pi t$'); a[1].set_xlabel('t'); a[1].set_ylabel('x')
for tv,col in zip([0.0,0.5,1.0,1.5],['#1f77b4','#ff7f0e','#2ca02c','#d62728']):
    with torch.no_grad(): us = u(xs,torch.full_like(xs,tv)).numpy().ravel()
    a[2].plot(xg,us,color=col,lw=2,label=f't={tv}')
    a[2].plot(xg,np.sin(PI*xg)*np.cos(PI*tv),'k--',lw=1,alpha=0.6)
a[2].set_title('snapshots: PINN (solid) vs exact (dashed)')
a[2].set_xlabel('x'); a[2].set_ylabel('u'); a[2].legend(ncol=2); a[2].grid(alpha=.3)
err_t = torch.sqrt(((U-Ue)**2).mean(0)/((Ue**2).mean(0)+1e-9)).numpy()
a[3].semilogy(tg,err_t,'r',lw=2)
a[3].set_title('rel $L_2$ error vs time (spikes = zero-crossings)')
a[3].set_xlabel('t'); a[3].grid(alpha=.3,which='both')
plt.tight_layout(); plt.savefig('wave_equation.png',bbox_inches='tight'); print('saved figure')

**What to take away.** The network reproduces the standing wave to a relative $L_2$ of about
$1.6\%$ across two full periods, and the snapshots sit on top of the exact solution. The apparent
error spikes at $t=0.5,1.5$ are an artefact of the *relative* norm dividing by near-zero energy at
the instants the string is flat---the absolute error there is small. Being oscillatory and second
order in time, the wave equation is a touch harder than the elliptic conduction problem, and a first
hint of the spectral-bias and causality issues taken up in Chapter 6.